### Evals for AI Engineers - Chap 2 LLMs and Eval Basics
- uv add litellm, phonenumbers, rank_bm25


#### Example: Email Summarizer

In [40]:
import os
import json
import numpy as np
from dotenv import load_dotenv
from IPython.display import Markdown, display
import phonenumbers

from litellm import embedding, completion

from rank_bm25 import BM25Okapi

In [41]:
email_text = """
From: Alice Johnson <alice.johnson@example.com>
To: Support Team

Hi,

Could you please reset my account password? I would also appreciate it if you could verify that my email address on file is correct.
I would also prefer to be contacted by phone. Call me at (415) 555-9021 ext. 7.

Thank you,
Alice Johnson
"""


#### A single LLM call

In [42]:
response = completion(model="openai/gpt-4.1", 
                      messages=[
                          {"role": "system","content": "You are an assistant that summarizes emails."},
                          {"role": "user","content": "Extract the sender's name and summarize the key requests:\n\n" + email_text}
                      ])
reply = response.choices[0].message.content
display(Markdown(reply))

Sender: Alice Johnson

Summary of key requests:
- Reset her account password.
- Verify that her email address on file is correct.
- Contact her by phone at (415) 555-9021 ext. 7 instead of email.

#### A conversation

In [43]:
messages = [
    {
        "role": "system",
        "content": (
            "You are an assistant that summarizes and "
            "answers questions about emails."
        ),
    },
    {"role": "user", "content": "Summarize this email:\n\n" + email_text},
]

response1 = completion(
    model="gpt-4o-2024-08-06",
    messages=messages,
)
print(response1["choices"][0]["message"]["content"])

# Add the model's reply to the conversation and continue
messages.append({
    "role": "assistant",
    "content": response1["choices"][0]["message"]["content"],
})
messages.append({
    "role": "user",
    "content": "Does she mention her preferred contact method?",
})

response2 = completion(
    model="gpt-4o-2024-08-06",
    messages=messages,
)

display(Markdown(response2["choices"][0]["message"]["content"]))


Alice Johnson requested a password reset for her account and asked for confirmation that her email address on file is correct. She prefers to be contacted by phone and provided her number as (415) 555-9021 ext. 7.


Yes, Alice Johnson mentions that she prefers to be contacted by phone. She provided her phone number for this purpose.

#### Retrieval

In [44]:
# Small archive of past support emails
past_emails = [
    "Ticket #4521: Please reset my password. I cannot log in.",
    "Ticket #4605: Could you verify my email address on file?",
    "Ticket #4710: I need help updating my phone number.",
]

In [45]:
# Cosine similarity scores two embedding vectors from -1 (opposite)
# to 1 (identical). Higher means the text is more similar.
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [46]:
# Embedding-based retrieval
corpus_response = embedding(
    model="text-embedding-3-small",
    input=past_emails,
)
corpus_embeddings = [item["embedding"] for item in corpus_response["data"]]

query_response = embedding(
    model="text-embedding-3-small",
    input=email_text,
)
query_embedding = query_response["data"][0]["embedding"]

scores = [cosine_similarity(query_embedding, e) for e in corpus_embeddings]
retrieved = past_emails[int(np.argmax(scores))]

# Combine retrieved context with the new email
messages = [
    {
        "role": "system",
        "content": "You are an assistant that summarizes emails with context.",
    },
    {
        "role": "user",
        "content": (
            f"Email:\n{email_text}\n\n"
            f"Related past email:\n{retrieved}"
        ),
    },
]

response = completion(
    model="gpt-4o-2024-08-06",
    messages=messages,
)

display(Markdown(response["choices"][0]["message"]["content"]))

Alice Johnson has requested a password reset for her account and would like to ensure that her email address on file is correct. Additionally, she prefers to be contacted by phone at (415) 555-9021 ext. 7 for further communication. This request relates to a previous inquiry (Ticket #4605) about email address verification.

- Keyword-based retrieval could be swapped in here using a library like whoosh or elasticsearch, returning the top BM25 match instead of computing embeddings.

#### Tool call
- normalizing phone number

In [47]:
# Define the tool as a Python function
def normalize_phone(number: str):
    parsed = phonenumbers.parse(number, "US")
    return phonenumbers.format_number(
        parsed, phonenumbers.PhoneNumberFormat.E164
    )

In [48]:
# Tell the model what the tool is and how to call it
# this is an example of a good tool spec

tools = [
    {"type": "function",
        "function": {
            "name": "normalize_phone",
            "description": "Convert a phone number to standardized E.164 format",
            "parameters": {
                "type": "object",
                "properties": {
                    "number": {
                        "type": "string",
                        "description": (
                            "Phone number to normalize, assumed to be US "
                            "if no country code is provided"
                        ),
                    }
                },
                "required": ["number"],
            },
        },
    }
]

In [49]:
messages = [
    {
        "role": "system",
        "content": (
            "You summarize emails. If the email contains a phone number, "
            "you MUST call the normalize_phone tool on it before writing the "
            "summary, and use the normalized number in the summary."
        ),
    },
    {
        "role": "user",
        "content": f"Summarize this email:\n\n{email_text}",
    },
]

resp = completion(
    model="gpt-4o-2024-08-06",
    messages=messages,
    tools=tools,
    # "auto" (the default) lets the model skip tools. "required" forces at least one tool call.
    # To force a specific tool: {"type": "function", "function": {"name": "normalize_phone"}}
    tool_choice="required",
)
msg = resp["choices"][0]["message"]

# When the model calls a tool, msg["content"] is usually None and the request
# lives in msg["tool_calls"] instead, so inspect both.
if msg.get("tool_calls"):
    for call in msg["tool_calls"]:
        print(f"Tool requested: {call['function']['name']}({call['function']['arguments']})")
else:
    display(Markdown(msg["content"]))

Tool requested: normalize_phone({"number":"(415) 555-9021 ext. 7"})


In [50]:

# If the model calls the tool, run it and send back the result
if msg.get("tool_calls"):
    # Append the assistant's message containing the tool calls
    messages.append(msg)
    print("Tool calls detected. Executing tool calls...")

    for call in msg["tool_calls"]:
        if call["function"]["name"] == "normalize_phone":
            args = json.loads(call["function"]["arguments"])
            result = normalize_phone(**args)
            messages.append({
                "role": "tool",
                "tool_call_id": call["id"],
                "name": "normalize_phone",
                "content": json.dumps(result),
            })

# Continue the conversation until the model produces a final summary
resp2 = completion(
    model="gpt-4o-2024-08-06",
    messages=messages,
    tools=tools,
)
display(Markdown(resp2["choices"][0]["message"]["content"]))

Tool calls detected. Executing tool calls...


Alice Johnson requests a password reset for her account and asks to verify her email address on file. She prefers to be contacted by phone at +14155559021.

#### Agent loop
- Minimal sketch of an agent loop

In [51]:
messages = [
    {
        "role": "system",
        "content": (
            "You are an agent that can retrieve past emails, "
            "call tools, and summarize."
        ),
    },
    {
        "role": "user",
        "content": f"Summarize this new email:\n\n{email_text}",
    },
]

for step in range(10):  # Cap the number of steps
    resp = completion(
        model="gpt-4o-2024-08-06",
        messages=messages,
        tools=tools,
        tool_choice="required",
    )
    msg = resp["choices"][0]["message"]

    # If the model requests a tool, run it and append the result
    if msg.get("tool_calls"):
        messages.append(msg)

        for call in msg["tool_calls"]:
            if call["function"]["name"] == "normalize_phone":
                args = json.loads(call["function"]["arguments"])
                tool_result = normalize_phone(**args)
            else:
                tool_result = "Unknown tool"

            messages.append({
                "role": "tool",
                "tool_call_id": call["id"],
                "name": call["function"]["name"],
                "content": json.dumps(tool_result),
            })
        continue

    # If the model produces a final answer, stop
    if msg.get("content"):
        print("Final output:", msg["content"])
        break

#### Propmpt template

In [52]:
display(Markdown(f"""
### Role
You are an assistant for our internal support team.
### Objective
Extract the sender’s name and summarize the key requests in an incoming email.
### Instructions
- Produce up to three concise bullet points for the requests.
- Do not include quoted text from prior threads or signatures.
- Keep the tone plain and professional.
### Context
[PASTE EMAIL TEXT HERE]
### Reasoning
Think through the steps needed to extract the sender and requests,
but return only the final formatted output.
### Output format
- Sender: <name>
- Requests:
- <bullet 1>
- <bullet 2>
- <bullet 3>
### Example
Input: "My account was locked after travel. Please reset it and
call me at (415) 555-9021. – Alice Johnson"
Output:
- Sender: Alice Johnson
- Requests:
- Reset account access
- Call back at (415) 555-9021
"""))


### Role
You are an assistant for our internal support team.
### Objective
Extract the sender’s name and summarize the key requests in an incoming email.
### Instructions
- Produce up to three concise bullet points for the requests.
- Do not include quoted text from prior threads or signatures.
- Keep the tone plain and professional.
### Context
[PASTE EMAIL TEXT HERE]
### Reasoning
Think through the steps needed to extract the sender and requests,
but return only the final formatted output.
### Output format
- Sender: <name>
- Requests:
- <bullet 1>
- <bullet 2>
- <bullet 3>
### Example
Input: "My account was locked after travel. Please reset it and
call me at (415) 555-9021. – Alice Johnson"
Output:
- Sender: Alice Johnson
- Requests:
- Reset account access
- Call back at (415) 555-9021


#### Retrieval 
- Keyword search

In [53]:
# Small archive of past emails
past_emails = [
    "Ticket #4521: Please reset my password. I cannot log in.",
    "Ticket #4605: Could you verify my email address on file?",
    "Ticket #4710: I need help updating my phone number.",
]

In [54]:

# Build BM25 index
tokenized_corpus = [doc.lower().split() for doc in past_emails]
print(tokenized_corpus)
bm25 = BM25Okapi(tokenized_corpus)

query = "I still can’t log in."
tokenized_query = query.lower().split()

# Score and select the best match
scores = bm25.get_scores(tokenized_query)
best_doc = past_emails[int(np.argmax(scores))]
print(best_doc)

# Example output:
# "Ticket #4521: Please reset my password. I cannot log in."

[['ticket', '#4521:', 'please', 'reset', 'my', 'password.', 'i', 'cannot', 'log', 'in.'], ['ticket', '#4605:', 'could', 'you', 'verify', 'my', 'email', 'address', 'on', 'file?'], ['ticket', '#4710:', 'i', 'need', 'help', 'updating', 'my', 'phone', 'number.']]
Ticket #4521: Please reset my password. I cannot log in.


#### ReAct Agent
- An agent is simply a system that loops through the following cycle: 
> - the model reasons about the current state, 
> - decides whether to act (by calling a tool or retrieving context), 
> - observes the result, 
> - and continues until it produces a final answer. 
- ReAct (short for “Reason + Act”; Yao et al. 2023).
- Modern LLM APIs now handle much of the ReAct loop internally; tool calling APIs automatically interleave reasoning and action without requiring explicit ReAct prompts. 
- The pattern is still useful to understand, because the underlying structure is the same.

In [55]:
display(Markdown(
    f"""
    ### Role
    You are an agent that helps summarize support emails.
    ### Objective
    Read a new email, use retrieval and tools when helpful, and produce
    a final summary for the support team.
    ### Instructions
    - You may retrieve past tickets if needed.
    - You may call `normalize_phone` when a phone number is present.
    - Work step by step. At each step, decide on one action
      (retrieve, call a tool, or stop with a final answer).
    - After you take an Action, you will receive an Observation,
      with the result. Use it to decide your next step.
    - End with a final answer in the specified format.
    - If the user’s request is ambiguous, [choose one]:
      **(a)** ask a clarifying question before proceeding;
      **(b)** infer the most likely intent and continue.
    ### Reasoning (ReAct pattern)
    Follow the Thought → Action → Observation cycle:
    - **Thought:** (optional, for development) a brief statement of what
      you intend to do.
    - **Action:** the tool to call, with JSON arguments.
    - **Observation:** will be provided after each action.
    - Repeat until you produce the final answer.
    ### Output format (for Final)
    Final answer:
    - Sender: <name>
    - Requests:
      - <bullet 1>
      - <bullet 2>
      - <bullet 3>
    - Phone: <normalized number if present>
    ### Step format example
    Thought: I should check prior tickets for context.
    Action: retrieve_tickets
    Args: {{"query": "can’t log in; password reset", "sender": "<sender_email>"}}
    Observation: [{{"id":"4521","text":"Please reset my password. I cannot log in."}}]
    Thought: A phone number is present; I will normalize it.
    Action: normalize_phone
    Args: {{"number": "(415) 555-9021 ext. 7"}}
    Observation: "+14155559021"
    Thought: I have enough information to write the summary.
    Final answer:
    - Sender: Alice Johnson
    - Requests:
      - Reset account access
      - Verify email on file
      - Call back at +14155559021
    - Phone: +14155559021
    """
))



    ### Role
    You are an agent that helps summarize support emails.
    ### Objective
    Read a new email, use retrieval and tools when helpful, and produce
    a final summary for the support team.
    ### Instructions
    - You may retrieve past tickets if needed.
    - You may call `normalize_phone` when a phone number is present.
    - Work step by step. At each step, decide on one action
      (retrieve, call a tool, or stop with a final answer).
    - After you take an Action, you will receive an Observation,
      with the result. Use it to decide your next step.
    - End with a final answer in the specified format.
    - If the user’s request is ambiguous, [choose one]:
      **(a)** ask a clarifying question before proceeding;
      **(b)** infer the most likely intent and continue.
    ### Reasoning (ReAct pattern)
    Follow the Thought → Action → Observation cycle:
    - **Thought:** (optional, for development) a brief statement of what
      you intend to do.
    - **Action:** the tool to call, with JSON arguments.
    - **Observation:** will be provided after each action.
    - Repeat until you produce the final answer.
    ### Output format (for Final)
    Final answer:
    - Sender: <name>
    - Requests:
      - <bullet 1>
      - <bullet 2>
      - <bullet 3>
    - Phone: <normalized number if present>
    ### Step format example
    Thought: I should check prior tickets for context.
    Action: retrieve_tickets
    Args: {"query": "can’t log in; password reset", "sender": "<sender_email>"}
    Observation: [{"id":"4521","text":"Please reset my password. I cannot log in."}]
    Thought: A phone number is present; I will normalize it.
    Action: normalize_phone
    Args: {"number": "(415) 555-9021 ext. 7"}
    Observation: "+14155559021"
    Thought: I have enough information to write the summary.
    Final answer:
    - Sender: Alice Johnson
    - Requests:
      - Reset account access
      - Verify email on file
      - Call back at +14155559021
    - Phone: +14155559021
    

#### Prompt caching

In [56]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 118
Output tokens: 68
Total tokens: 186
Cached tokens: 0
Total cost: 0.0975 cents
